# Módulo 39: SQL avanzado

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m39-sql-avanzado.md](m39-sql-avanzado.md). No es una tarea entregada del programa.

Muchas preguntas no se responden con una sola consulta plana: "¿qué productos se han vendido alguna vez?", "¿en qué departamento está *hoy* cada empleado?", "¿cuáles órdenes son de 2011?". Para eso se encadenan pasos: una consulta dentro de otra, resultados intermedios con nombre o tablas de trabajo temporales. También veremos las funciones de fecha y de texto más usadas.

1. Subconsultas con `IN`
2. `EXISTS` y subconsultas correlacionadas
3. Joins de varias tablas y tablas de historial
4. CTE: consultas con nombre (`WITH`)
5. Tablas temporales e `INSERT ... SELECT`
6. Funciones de fecha
7. Funciones de texto y `CAST`
8. Errores frecuentes
9. Ejercicio

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE Product (ProductID INTEGER PRIMARY KEY, Name TEXT, ListPrice REAL);
CREATE TABLE SalesOrderHeader (
    SalesOrderID INTEGER PRIMARY KEY, SalesOrderNumber TEXT,
    PurchaseOrderNumber TEXT, OrderDate TEXT
);
CREATE TABLE SalesOrderDetail (SalesOrderID INTEGER, ProductID INTEGER, OrderQty INTEGER);

INSERT INTO Product VALUES
 (707, 'Sport-100 Helmet, Red', 34.99), (708, 'Sport-100 Helmet, Black', 34.99),
 (709, 'Mountain Bike Socks, M', 9.50), (711, 'Sport-100 Helmet, Blue', 34.99),
 (712, 'AWC Logo Cap', 8.99),          (714, 'Long-Sleeve Logo Jersey, M', 49.99),
 (715, 'Long-Sleeve Logo Jersey, L', 49.99), (716, 'Long-Sleeve Logo Jersey, XL', 49.99);
INSERT INTO SalesOrderHeader VALUES
 (43659, 'SO43659', 'PO522145787',   '2011-05-31'),
 (43660, 'SO43660', 'PO18850127500', '2011-05-31'),
 (51721, 'SO51721', NULL,            '2012-03-30'),
 (63289, 'SO63289', 'PO8961116429',  '2013-01-02');
INSERT INTO SalesOrderDetail VALUES
 (43659, 707, 1), (43659, 712, 2), (43660, 714, 1),
 (51721, 707, 3), (63289, 711, 1), (63289, 712, 1);
""")


def q(sql):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)

## 1. Subconsultas con `IN`

Una **subconsulta** es un `SELECT` escrito dentro de otra consulta. Con `IN`, la subconsulta devuelve una lista de valores y la consulta externa conserva las filas cuyo valor está en esa lista.

Esta subconsulta es **no correlacionada**: no depende de la fila externa, así que se puede evaluar una sola vez y su resultado sirve para todas las filas. Con `NOT IN` se obtiene lo contrario: los productos que nunca se han vendido.

In [2]:
# Productos que aparecen en al menos una línea de orden
display(q("""
SELECT ProductID, Name
FROM Product
WHERE ProductID IN (SELECT ProductID FROM SalesOrderDetail)
ORDER BY Name
"""))

# Productos que nunca se han vendido
q("""
SELECT ProductID, Name
FROM Product
WHERE ProductID NOT IN (SELECT ProductID FROM SalesOrderDetail)
ORDER BY Name
""")

,ProductID,Name
0,712,AWC Logo Cap
1,714,"Long-Sleeve Logo Jersey, M"
2,711,"Sport-100 Helmet, Blue"
3,707,"Sport-100 Helmet, Red"


,ProductID,Name
0,715,"Long-Sleeve Logo Jersey, L"
1,716,"Long-Sleeve Logo Jersey, XL"
2,709,"Mountain Bike Socks, M"
3,708,"Sport-100 Helmet, Black"


## 2. `EXISTS` y subconsultas correlacionadas

`EXISTS (subconsulta)` es verdadero si la subconsulta devuelve **al menos una fila**; no importa qué columnas devuelva, por eso es habitual escribir `SELECT 1`.

La subconsulta suele ser **correlacionada**: hace referencia a una columna de la consulta externa (`p.ProductID`), así que conceptualmente se evalúa una vez por cada fila de afuera.

| Forma | Pregunta | Nombre habitual |
| --- | --- | --- |
| `WHERE x IN (SELECT ...)` / `WHERE EXISTS (...)` | ¿Tiene al menos una pareja? | Semi-join |
| `WHERE x NOT IN (SELECT ...)` / `WHERE NOT EXISTS (...)` | ¿No tiene ninguna? | Anti-join |

In [3]:
# Productos vendidos en alguna orden de 2011 (correlacionada con p.ProductID)
q("""
SELECT p.ProductID, p.Name
FROM Product p
WHERE EXISTS (
    SELECT 1
    FROM SalesOrderDetail d
    JOIN SalesOrderHeader h ON h.SalesOrderID = d.SalesOrderID
    WHERE d.ProductID = p.ProductID
      AND h.OrderDate >= '2011-01-01' AND h.OrderDate < '2012-01-01'
)
""")

,ProductID,Name
0,707,"Sport-100 Helmet, Red"
1,712,AWC Logo Cap
2,714,"Long-Sleeve Logo Jersey, M"


A diferencia de un `JOIN`, `EXISTS` no repite filas: cada producto aparece una sola vez, tenga una o muchas líneas que cumplan la condición.

## 3. Joins de varias tablas y tablas de historial

Un `JOIN` puede encadenar tantas tablas como haga falta: cada `JOIN ... ON` agrega una tabla al resultado anterior.

Muchas bases guardan el **historial** de una relación en lugar de solo su estado actual. Por ejemplo, `EmployeeDepartmentHistory` tiene una fila por cada periodo que un empleado pasó en un departamento, con `StartDate` y `EndDate`. El periodo **vigente** es el que todavía no termina, es decir, el que tiene `EndDate IS NULL`.

In [4]:
conn.executescript("""
CREATE TABLE Person (BusinessEntityID INTEGER PRIMARY KEY, FirstName TEXT, LastName TEXT);
CREATE TABLE Employee (BusinessEntityID INTEGER PRIMARY KEY, JobTitle TEXT, HireDate TEXT);
CREATE TABLE Department (DepartmentID INTEGER PRIMARY KEY, Name TEXT);
CREATE TABLE EmployeeDepartmentHistory (
    BusinessEntityID INTEGER, DepartmentID INTEGER, StartDate TEXT, EndDate TEXT
);
INSERT INTO Person VALUES
 (1, 'Ken', 'Sánchez'), (2, 'Terri', 'Duffy'), (3, 'Roberto', 'Tamburello'),
 (4, 'Rob', 'Walters'), (5, 'Gail', 'Erickson'), (6, 'Jossef', 'Goldberg'),
 (7, 'Dylan', 'Miller'), (8, 'Linda', 'Mitchell');
INSERT INTO Employee VALUES
 (1, 'Chief Executive Officer', '2009-01-14'), (2, 'Vice President of Engineering', '2008-01-31'),
 (3, 'Engineering Manager', '2007-11-11'),     (4, 'Senior Tool Designer', '2007-12-05'),
 (5, 'Design Engineer', '2008-01-06'),         (6, 'Design Engineer', '2008-01-24'),
 (7, 'Research and Development Manager', '2009-02-08');
INSERT INTO Department VALUES
 (1, 'Engineering'), (2, 'Tool Design'), (3, 'Sales'), (4, 'Marketing'), (5, 'Research and Development');
INSERT INTO EmployeeDepartmentHistory VALUES
 (1, 1, '2009-01-14', NULL),
 (2, 1, '2008-01-31', NULL),
 (3, 2, '2007-11-11', '2009-12-31'), (3, 1, '2010-01-01', NULL),
 (4, 2, '2007-12-05', NULL),
 (5, 3, '2008-01-06', '2010-06-30'), (5, 4, '2010-07-01', NULL),
 (6, 3, '2008-01-24', NULL),
 (7, 5, '2009-02-08', NULL);
""")

# Departamento actual de cada empleado: cuatro tablas y el filtro de vigencia
q("""
SELECT e.BusinessEntityID,
       p.FirstName || ' ' || p.LastName AS Empleado,
       d.Name                           AS Departamento
FROM Employee e
JOIN Person p                     ON p.BusinessEntityID = e.BusinessEntityID
JOIN EmployeeDepartmentHistory h  ON h.BusinessEntityID = e.BusinessEntityID
JOIN Department d                 ON d.DepartmentID = h.DepartmentID
WHERE h.EndDate IS NULL
ORDER BY e.BusinessEntityID
""")

,BusinessEntityID,Empleado,Departamento
0,1,Ken Sánchez,Engineering
1,2,Terri Duffy,Engineering
2,3,Roberto Tamburello,Engineering
3,4,Rob Walters,Tool Design
4,5,Gail Erickson,Marketing
5,6,Jossef Goldberg,Sales
6,7,Dylan Miller,Research and Development


En T-SQL la concatenación usa `+`: `p.FirstName + ' ' + p.LastName AS Empleado`. Linda Mitchell (persona 8) no aparece porque no es empleada: el `JOIN` con `Employee` la descarta.

## 4. CTE: consultas con nombre (`WITH`)

Un **CTE** (*Common Table Expression*) da nombre a una consulta intermedia que se usa enseguida, en la misma sentencia. Hace lo mismo que una subconsulta en `FROM`, pero se lee de arriba abajo, como una receta en pasos.

```sql
WITH paso1 AS (SELECT ...),
     paso2 AS (SELECT ... FROM paso1 ...)
SELECT ... FROM paso2;
```

La sintaxis es igual en SQLite y en SQL Server. En T-SQL, si la sentencia anterior no termina en `;`, el `WITH` debe ir precedido de punto y coma.

In [5]:
q("""
WITH Vigente AS (
    SELECT BusinessEntityID, DepartmentID
    FROM EmployeeDepartmentHistory
    WHERE EndDate IS NULL
),
ConNombre AS (
    SELECT v.BusinessEntityID, p.FirstName || ' ' || p.LastName AS Empleado, d.Name AS Departamento
    FROM Vigente v
    JOIN Person p     ON p.BusinessEntityID = v.BusinessEntityID
    JOIN Department d ON d.DepartmentID = v.DepartmentID
)
SELECT Departamento, COUNT(*) AS empleados
FROM ConNombre
GROUP BY Departamento
ORDER BY empleados DESC, Departamento
""")

,Departamento,empleados
0,Engineering,3
1,Marketing,1
2,Research and Development,1
3,Sales,1
4,Tool Design,1


## 5. Tablas temporales e `INSERT ... SELECT`

Una **tabla temporal** guarda un resultado intermedio para usarlo en varias consultas posteriores, y desaparece sola al cerrar la sesión. A diferencia del CTE, que solo existe dentro de una sentencia, la tabla temporal persiste mientras dure la conexión.

| Motor | Sintaxis | Visible para | Se elimina |
| --- | --- | --- | --- |
| SQL Server | `CREATE TABLE #Local (...)` | Solo la sesión que la creó | Al cerrar esa sesión |
| SQL Server | `CREATE TABLE ##Global (...)` | Todas las sesiones | Cuando se cierra la sesión que la creó y ninguna otra la está usando |
| SQLite | `CREATE TEMP TABLE t (...)` | Solo la conexión que la creó | Al cerrar la conexión |

`INSERT INTO destino (columnas) SELECT ...` inserta en una tabla el resultado de una consulta, con todos sus filtros.

In [6]:
conn.execute("CREATE TEMP TABLE ProductosVendidos (ProductID INTEGER, Name TEXT)")
cur = conn.execute("""
INSERT INTO ProductosVendidos (ProductID, Name)
SELECT ProductID, Name
FROM Product
WHERE ProductID IN (SELECT ProductID FROM SalesOrderDetail)
""")
print("Filas insertadas:", cur.rowcount)
q("SELECT * FROM ProductosVendidos ORDER BY ProductID")

Filas insertadas: 4


,ProductID,Name
0,707,"Sport-100 Helmet, Red"
1,711,"Sport-100 Helmet, Blue"
2,712,AWC Logo Cap
3,714,"Long-Sleeve Logo Jersey, M"


En T-SQL, el mismo paso con una tabla local y otra global sería:

```sql
CREATE TABLE #ProductosVendidos  (ProductID INT, Name NVARCHAR(50));
CREATE TABLE ##ProductosVendidos (ProductID INT, Name NVARCHAR(50));

INSERT INTO #ProductosVendidos (ProductID, Name)
SELECT ProductID, Name
FROM Production.Product
WHERE ProductID IN (SELECT ProductID FROM Sales.SalesOrderDetail);
```

Para ver el alcance en SQLite, abrimos dos conexiones a una misma base en memoria compartida: la segunda ve las tablas normales, pero no la temporal de la primera.

In [7]:
uri = "file:alcance?mode=memory&cache=shared"   # base en memoria compartida, sin archivo
a = sqlite3.connect(uri, uri=True)
b = sqlite3.connect(uri, uri=True)

a.execute("CREATE TABLE Normal (x INTEGER)")
a.execute("CREATE TEMP TABLE Temporal (x INTEGER)")

for nombre in ["Normal", "Temporal"]:
    try:
        b.execute(f"SELECT COUNT(*) FROM {nombre}")
        print(f"La conexión b ve la tabla {nombre}")
    except sqlite3.OperationalError as error:
        print(f"La conexión b no ve la tabla {nombre}: {error}")

a.close()
b.close()

La conexión b ve la tabla Normal
La conexión b no ve la tabla Temporal: no such table: Temporal


## 6. Funciones de fecha

SQLite guarda las fechas como texto ISO y las procesa con funciones; SQL Server tiene tipos de fecha y funciones propias:

| Tarea | T-SQL | SQLite |
| --- | --- | --- |
| Año | `YEAR(d)` | `CAST(strftime('%Y', d) AS INTEGER)` |
| Mes | `MONTH(d)` | `CAST(strftime('%m', d) AS INTEGER)` |
| Sumar días | `DATEADD(day, 30, d)` | `date(d, '+30 days')` |
| Días entre fechas | `DATEDIFF(day, d1, d2)` | `julianday(d2) - julianday(d1)` |
| Hoy | `GETDATE()` | `date('now')` |

`strftime` devuelve **texto**: `strftime('%Y', d) = 2011` compara texto con número y no coincide. Hay que comparar con `'2011'` o convertir con `CAST`.

In [8]:
q("""
SELECT SalesOrderNumber, OrderDate,
       CAST(strftime('%Y', OrderDate) AS INTEGER)         AS anio,
       date(OrderDate, '+30 days')                         AS vence,
       CAST(julianday('2013-12-31') - julianday(OrderDate) AS INTEGER) AS dias_hasta_fin_2013
FROM SalesOrderHeader
""")

,SalesOrderNumber,OrderDate,anio,vence,dias_hasta_fin_2013
0,SO43659,2011-05-31,2011,2011-06-30,945
1,SO43660,2011-05-31,2011,2011-06-30,945
2,SO51721,2012-03-30,2012,2012-04-29,641
3,SO63289,2013-01-02,2013,2013-02-01,363


In [9]:
# Órdenes de 2011, de dos formas equivalentes
display(q("SELECT SalesOrderNumber FROM SalesOrderHeader WHERE strftime('%Y', OrderDate) = '2011'"))
q("SELECT SalesOrderNumber FROM SalesOrderHeader WHERE OrderDate >= '2011-01-01' AND OrderDate < '2012-01-01'")

,SalesOrderNumber
0,SO43659
1,SO43660


,SalesOrderNumber
0,SO43659
1,SO43660


La segunda forma, con un rango, es preferible en tablas grandes: al no aplicar una función a la columna, el motor puede usar un índice sobre `OrderDate`. En T-SQL ocurre lo mismo con `YEAR(OrderDate) = 2011` frente al rango.

## 7. Funciones de texto y `CAST`

| Tarea | T-SQL | SQLite |
| --- | --- | --- |
| Subcadena desde la posición *i*, *n* caracteres | `SUBSTRING(s, i, n)` | `substr(s, i, n)` |
| Desde la posición *i* hasta el final | `SUBSTRING(s, i, LEN(s))` | `substr(s, i)` |
| Longitud | `LEN(s)` | `length(s)` |
| Mayúsculas | `UPPER(s)` | `upper(s)` |
| Reemplazar | `REPLACE(s, a, b)` | `replace(s, a, b)` |
| Convertir tipo | `CAST(x AS VARCHAR(10))` | `CAST(x AS TEXT)` |

Las posiciones empiezan en 1. Quitar los dos primeros caracteres equivale a empezar en la posición 3.

In [10]:
q("""
SELECT SalesOrderNumber,
       substr(SalesOrderNumber, 3)                          AS sin_prefijo,
       PurchaseOrderNumber,
       substr(PurchaseOrderNumber, 3, length(PurchaseOrderNumber)) AS po_sin_prefijo,
       length(SalesOrderNumber)                             AS largo
FROM SalesOrderHeader
""")

,SalesOrderNumber,sin_prefijo,PurchaseOrderNumber,po_sin_prefijo,largo
0,SO43659,43659,PO522145787,522145787,7
1,SO43660,43660,PO18850127500,18850127500,7
2,SO51721,51721,None,None,7
3,SO63289,63289,PO8961116429,8961116429,7


`substr` de un `NULL` devuelve `NULL`, como la orden en línea 51721, que no tiene número de compra.

Las funciones de texto tienen sentido sobre **códigos de texto** como `'SO43659'` o `'PO522145787'`, cuyo prefijo significa algo. Un identificador numérico como `SalesOrderID` no tiene prefijo: convertirlo a texto y recortarlo produce un número distinto sin significado.

In [11]:
q("""
SELECT SalesOrderID,
       substr(CAST(SalesOrderID AS TEXT), 3) AS id_recortado,      -- sin sentido
       SalesOrderNumber,
       substr(SalesOrderNumber, 3)           AS numero_sin_prefijo  -- correcto
FROM SalesOrderHeader
""")

,SalesOrderID,id_recortado,SalesOrderNumber,numero_sin_prefijo
0,43659,659,SO43659,43659
1,43660,660,SO43660,43660
2,51721,721,SO51721,51721
3,63289,289,SO63289,63289


## 8. Errores frecuentes

**`NOT IN` con una subconsulta que contiene `NULL`.** `x NOT IN (1, 2, NULL)` nunca es verdadero, porque `x <> NULL` es desconocido. Si la subconsulta puede devolver nulos, el resultado queda vacío sin ningún aviso. `NOT EXISTS` no tiene ese problema.

In [12]:
conn.executescript("""
CREATE TABLE Customer (CustomerID INTEGER PRIMARY KEY, PersonID INTEGER);
INSERT INTO Customer VALUES (11000, 1), (11001, 2), (11002, NULL);   -- 11002 es una tienda
""")

# Personas que no son clientes
display(q("SELECT BusinessEntityID FROM Person WHERE BusinessEntityID NOT IN (SELECT PersonID FROM Customer)"))
q("""
SELECT p.BusinessEntityID
FROM Person p
WHERE NOT EXISTS (SELECT 1 FROM Customer c WHERE c.PersonID = p.BusinessEntityID)
""")

,BusinessEntityID


,BusinessEntityID
0,3
1,4
2,5
3,6
4,7
5,8


**Olvidar el filtro de vigencia en una tabla de historial.** Sin `EndDate IS NULL`, un empleado que cambió de departamento aparece una vez por cada periodo:

In [13]:
q("""
SELECT h.BusinessEntityID, d.Name, h.StartDate, h.EndDate
FROM EmployeeDepartmentHistory h
JOIN Department d ON d.DepartmentID = h.DepartmentID
WHERE h.BusinessEntityID IN (3, 5)
""")

,BusinessEntityID,Name,StartDate,EndDate
0,3,Tool Design,2007-11-11,2009-12-31
1,3,Engineering,2010-01-01,None
2,5,Sales,2008-01-06,2010-06-30
3,5,Marketing,2010-07-01,None


**Aplicar funciones de texto a un identificador numérico.** Como se vio en la sección 7, si el dato que se quiere recortar es un código con prefijo, hay que buscar la columna de texto que lo contiene en lugar de convertir el ID.

**Comparar `strftime` con un número.** `strftime('%Y', d) = 2011` devuelve cero filas porque compara el texto `'2011'` con el entero 2011:

In [14]:
q("SELECT COUNT(*) AS filas FROM SalesOrderHeader WHERE strftime('%Y', OrderDate) = 2011")

,filas
0,0


## 9. Ejercicio

Una academia registra cursos, estudiantes e inscripciones. Una inscripción vigente tiene `FechaBaja` nula:

```python
cursos = [("CUR-101", "SQL"), ("CUR-102", "Python"), ("CUR-103", "Power BI")]
estudiantes = [(1, "Ana"), (2, "Luis"), (3, "Marta"), (4, "Pedro"), (5, "Sofía")]
inscripciones = [  # (EstudianteID, CursoID, FechaAlta, FechaBaja)
    (1, "CUR-101", "2024-01-10", None),
    (1, "CUR-102", "2023-09-01", "2023-12-15"),
    (2, "CUR-101", "2024-02-05", None),
    (3, "CUR-102", "2024-03-01", None),
    (3, "CUR-101", "2023-05-20", "2023-08-30"),
    (4, "CUR-103", "2023-11-11", "2024-01-31"),
]
```

1. Lista los estudiantes que tienen al menos una inscripción vigente, usando `EXISTS`.
2. Lista los estudiantes que nunca se han inscrito, usando `NOT EXISTS`.
3. Con un CTE, cuenta los estudiantes vigentes por curso y muestra todos los cursos, incluso los que tienen cero.
4. Crea una tabla temporal con las altas de 2024 (estudiante, curso y número de curso sin el prefijo `'CUR-'`) usando `INSERT ... SELECT`.

### Solución

In [15]:
cursos = [("CUR-101", "SQL"), ("CUR-102", "Python"), ("CUR-103", "Power BI")]
estudiantes = [(1, "Ana"), (2, "Luis"), (3, "Marta"), (4, "Pedro"), (5, "Sofía")]
inscripciones = [
    (1, "CUR-101", "2024-01-10", None),
    (1, "CUR-102", "2023-09-01", "2023-12-15"),
    (2, "CUR-101", "2024-02-05", None),
    (3, "CUR-102", "2024-03-01", None),
    (3, "CUR-101", "2023-05-20", "2023-08-30"),
    (4, "CUR-103", "2023-11-11", "2024-01-31"),
]
conn.execute("CREATE TABLE Curso (CursoID TEXT PRIMARY KEY, Nombre TEXT)")
conn.execute("CREATE TABLE Estudiante (EstudianteID INTEGER PRIMARY KEY, Nombre TEXT)")
conn.execute("CREATE TABLE Inscripcion (EstudianteID INTEGER, CursoID TEXT, FechaAlta TEXT, FechaBaja TEXT)")
conn.executemany("INSERT INTO Curso VALUES (?, ?)", cursos)
conn.executemany("INSERT INTO Estudiante VALUES (?, ?)", estudiantes)
conn.executemany("INSERT INTO Inscripcion VALUES (?, ?, ?, ?)", inscripciones)

display(q("""
SELECT e.Nombre FROM Estudiante e
WHERE EXISTS (SELECT 1 FROM Inscripcion i
              WHERE i.EstudianteID = e.EstudianteID AND i.FechaBaja IS NULL)
"""))
q("""
SELECT e.Nombre FROM Estudiante e
WHERE NOT EXISTS (SELECT 1 FROM Inscripcion i WHERE i.EstudianteID = e.EstudianteID)
""")

,Nombre
0,Ana
1,Luis
2,Marta


,Nombre
0,Sofía


Pedro no aparece en la primera lista (su única inscripción terminó) ni en la segunda (sí se inscribió alguna vez). Sofía es la única que nunca se inscribió.

In [16]:
q("""
WITH Vigentes AS (
    SELECT CursoID, COUNT(*) AS estudiantes
    FROM Inscripcion
    WHERE FechaBaja IS NULL
    GROUP BY CursoID
)
SELECT c.CursoID, c.Nombre, COALESCE(v.estudiantes, 0) AS estudiantes_vigentes
FROM Curso c
LEFT JOIN Vigentes v ON v.CursoID = c.CursoID
ORDER BY c.CursoID
""")

,CursoID,Nombre,estudiantes_vigentes
0,CUR-101,SQL,2
1,CUR-102,Python,1
2,CUR-103,Power BI,0


In [17]:
conn.execute("CREATE TEMP TABLE Altas2024 (Estudiante TEXT, CursoID TEXT, NumeroCurso INTEGER)")
conn.execute("""
INSERT INTO Altas2024 (Estudiante, CursoID, NumeroCurso)
SELECT e.Nombre, i.CursoID, CAST(substr(i.CursoID, 5) AS INTEGER)
FROM Inscripcion i
JOIN Estudiante e ON e.EstudianteID = i.EstudianteID
WHERE i.FechaAlta >= '2024-01-01' AND i.FechaAlta < '2025-01-01'
""")
q("SELECT * FROM Altas2024")

,Estudiante,CursoID,NumeroCurso
0,Ana,CUR-101,101
1,Luis,CUR-101,101
2,Marta,CUR-102,102


In [18]:
conn.close()

## Resumen

- Una subconsulta con `IN` filtra contra una lista; `EXISTS` y `NOT EXISTS` (semi-join y anti-join) preguntan si hay pareja y no fallan con `NULL`.
- En tablas de historial, el registro vigente es el que tiene `EndDate IS NULL`; sin ese filtro, las filas se duplican.
- Un CTE (`WITH`) nombra pasos intermedios dentro de una sentencia; una tabla temporal los conserva durante la sesión (`#local` y `##global` en T-SQL, `TEMP` en SQLite).
- `INSERT ... SELECT` guarda en una tabla el resultado de una consulta con todos sus filtros.
- Las funciones de fecha y texto difieren entre motores (`YEAR`/`strftime`, `SUBSTRING`/`substr`); se aplican a códigos de texto, no a identificadores numéricos.

En el **Módulo 40** se trabaja con SQLite desde Python: conexiones, cursores, parámetros y transacciones.